# HC-FALCON: tính embedding trên GPU (Colab / Kaggle)

Máy local không có GPU, nên mọi bước embedding chạy ở đây rồi tải kết quả về.

1. **Runtime → Change runtime type → GPU** (T4 là đủ).
2. Chạy lần lượt các ô của chế độ bạn cần.
3. Tải file zip ở ô cuối, giải nén vào `data/embeddings/` trong repo local.

| Chế độ | Khi nào | Kết quả |
|---|---|---|
| **(a) Tái lập FALCON** trên Defects4J | Tuần 2 | `d4j/<Project>/unixcoder_test_embeddings.npy`, thứ tự do code FALCON gốc sinh ra, APFD của artifact |
| **(b) LRTS** (masked mean-pool từng method → vector class) | Tuần 4 | sẽ bổ sung |

Trên Kaggle: bật GPU trong *Settings → Accelerator*, đổi `BASE` thành `/kaggle/working`, tải zip ở tab *Output*.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch, transformers
print('torch', torch.__version__, '| transformers', transformers.__version__, '| cuda', torch.cuda.is_available())

## Chế độ (a): tái lập FALCON trên 6 dự án Defects4J

Dùng **nguyên code của artifact FALCON** (`py/embedding_generator.py`, `py/submodtcp.py`, `py/metric.py`) để embedding và thứ tự trùng với cách tác giả làm. Lưu ý: generator của FALCON lấy trung bình cả token padding (không dùng attention mask), batch 16, max 512 token. Ta giữ nguyên để tái lập; chế độ (b) sẽ dùng masked mean-pool.

In [ ]:
BASE = '/content'   # Kaggle: '/kaggle/working'
PROJECTS = ['Chart', 'Closure', 'Lang', 'Math', 'Mockito', 'Time']
MODEL = 'unixcoder'
FALCON_REPO = 'https://github.com/anonymous-ing/falcon-icst-2026.git'

In [ ]:
import os, subprocess
os.chdir(BASE)
if not os.path.isdir('falcon-icst-2026'):
    subprocess.run(['git', 'clone', '--depth', '1', FALCON_REPO], check=True)
FALCON_COMMIT = subprocess.run(['git', '-C', 'falcon-icst-2026', 'rev-parse', 'HEAD'],
                               capture_output=True, text=True, check=True).stdout.strip()
print('FALCON artifact commit:', FALCON_COMMIT)

In [ ]:
# Sinh embedding bằng script gốc của FALCON (mỗi dự án vài chục giây trên T4)
os.chdir(f'{BASE}/falcon-icst-2026')
for P in PROJECTS:
    print('=' * 20, P)
    r = subprocess.run(['python', 'py/embedding_generator.py', '--project', P,
                        '--models', MODEL, '--device', 'cuda'], capture_output=True, text=True)
    print(r.stdout[-600:])
    if r.returncode != 0:
        print(r.stderr[-2000:])
        raise RuntimeError(f'embedding failed for {P}')

In [ ]:
# Chạy FacilityLocation.lazy_greedy của artifact, lưu thứ tự + APFD để so với bản cài đặt local
import sys, json, time, shutil
import numpy as np
sys.path.insert(0, 'py')
import submodtcp, metric

EXPORT = f'{BASE}/export/d4j'
summary = []
for P in PROJECTS:
    emb_file = f'input/{P}/embeddings/{MODEL}_test_embeddings.npy'
    emb = np.load(emb_file)
    t0 = time.perf_counter()
    fl = submodtcp.FacilityLocation(source_embeddings=emb, metric='cosine', device='cuda')
    order, _ = fl.lazy_greedy(budget=emb.shape[0])
    elapsed = time.perf_counter() - t0
    apfd = metric.apfd(order, f'input/{P}/fault_matrix.pickle')
    out = f'{EXPORT}/{P}'
    os.makedirs(out, exist_ok=True)
    shutil.copy(emb_file, out)
    shutil.copy(f'input/{P}/embeddings/{MODEL}_timing.json', out)
    np.save(f'{out}/falcon_artifact_order.npy', np.asarray(order, dtype=np.int64))
    rec = dict(project=P, n=int(emb.shape[0]), dim=int(emb.shape[1]), apfd=float(apfd),
               prioritization_time_s=round(elapsed, 3), device='cuda',
               gpu=torch.cuda.get_device_name(0), falcon_commit=FALCON_COMMIT)
    with open(f'{out}/falcon_artifact_result.json', 'w') as fh:
        json.dump(rec, fh, indent=2)
    summary.append(rec)
    print(rec)

print('median APFD:', float(np.median([r['apfd'] for r in summary])), '(paper: 0.731)')

In [ ]:
# Đóng gói và tải về. Local: giải nén vào <repo>/data/embeddings/  -> data/embeddings/d4j/<Project>/...
archive = shutil.make_archive(f'{BASE}/d4j_export', 'zip', f'{BASE}/export')
print(archive, round(os.path.getsize(archive) / 1e6, 1), 'MB')
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print('Không phải Colab: tải file ở tab Output (Kaggle).')

## Chế độ (b): LRTS — sẽ bổ sung ở tuần 4

Đầu vào là corpus blob (`data/lrts/blobs/<project>.jsonl`, mỗi dòng một phiên bản file test với danh sách method) do `src/hcfalcon/lrts/export_blobs.py` sinh ra. Mỗi method được embed bằng `microsoft/unixcoder-base` với **masked** mean-pool, chuẩn hóa L2, rồi lấy trung bình thành vector của class. Kết quả: `data/embeddings/lrts/<project>/unixcoder.npz` (khóa = blob SHA).